In [4]:
def fetch_bars(symbol, timeframe, count, before=None):
        import pandas as pd
        # yfinance optionally imports `curl_cffi.requests` if curl_cffi is
        # installed. In some environments curl_cffi imports eventlet which
        # pulls in green socket wrappers that lack `sendmsg`, causing
        # an AttributeError during import (see GreenSocket.sendmsg). To
        # avoid that, pre-register a lightweight shim module named
        # `curl_cffi` that exposes the standard `requests` API. This
        # prevents yfinance from importing the real curl_cffi package.
        try:
            import sys, types
            import requests as _real_requests
            if 'curl_cffi' not in sys.modules:
                # Create a requests-like shim that tolerates the
                # `impersonate` kwarg used by yfinance when curl_cffi
                # is present. We subclass the real requests.Session to
                # accept and ignore that extra kwarg.
                class _ShimSession(_real_requests.Session):
                    def __init__(self, *args, **kwargs):
                        kwargs.pop('impersonate', None)
                        super().__init__(*args, **kwargs)

                shim = types.ModuleType('curl_cffi')
                shim.requests = types.ModuleType('curl_cffi.requests')
                shim.requests.Session = _ShimSession
                # expose top-level helpers
                shim.requests.get = _real_requests.get
                shim.requests.post = _real_requests.post
                shim.requests.request = _real_requests.request
                shim.requests.utils = getattr(_real_requests, 'utils', None)
                shim.requests.Response = getattr(_real_requests, 'Response', None)
                shim.requests.exceptions = getattr(_real_requests, 'exceptions', None)
                shim.requests.adapters = getattr(_real_requests, 'adapters', None)
                shim.requests.models = getattr(_real_requests, 'models', None)
                # requests.sessions module contains the Session class reference
                shim.requests.session = getattr(_real_requests, 'sessions', types.SimpleNamespace(Session=_ShimSession))
                sys.modules['curl_cffi'] = shim
        except Exception:
            # If anything goes wrong here, fall back and let yfinance
            # import proceed (it may raise the original error).
            pass

        import yfinance as yf
        
        yf_symbol = config.SYMBOL_MAP.get(symbol, symbol)
        
        # yfinance timeframe mapping (flexible for case and format)
        yf_tf_map = {
            "M1": "1m", "M5": "5m", "M15": "15m",
            "H1": "1h", "H4": "4h", "D1": "1d",
            "4H": "4h", "1H": "1h"
        }
        interval = yf_tf_map.get(timeframe.upper(), "1h")
        cache_path = self._get_cache_path(symbol, timeframe)
        lock_path = cache_path + ".lock"
        
        # Ensure cache dir exists
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)

        with portalocker.Lock(lock_path, timeout=10) as fh:
            df = pd.DataFrame()
            if os.path.exists(cache_path):
                try:
                    df = pd.read_parquet(cache_path)
                except Exception as e:
                    print(f"⚠️ Cache read error for {symbol}: {e}. Rebuilding...")

            # Determine if we need to download more
            needs_download = False
            now_utc = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
            
            if df.empty:
                needs_download = True
            elif not before: # Check if we need latest (1h heuristic)
                last_dt = df["datetime"].max()
                # Use now_utc for comparison to match tz-naive UTC stored in parquet
                if (now_utc - last_dt).total_seconds() > 3600:
                    needs_download = True
            
            if needs_download:
                # Extra guard: if last_dt is somehow after now_utc, don't request from yfinance
                if not df.empty and df["datetime"].max() >= now_utc:
                    print(f"ℹ️ Cache for {symbol} is already up-to-date or ahead ({df['datetime'].max()}). Skipping download.")
                    needs_download = False
                print(f"📡 Downloading {yf_symbol} ({interval}) from yfinance...")
                ticker = yf.Ticker(yf_symbol)
                try:
                    # Try a sequence of intervals: preferred interval first,
                    # then fallbacks. This helps when Yahoo doesn't provide
                    # data for larger intervals like 4h for some tickers.
                    alt_intervals = {
                        'H4': ['1h', '30m'],
                        'H1': ['30m', '15m'],
                        'D1': ['1h'],
                    }

                    attempt_intervals = [interval]
                    attempt_intervals += alt_intervals.get(timeframe.upper(), [])

                    new_df = None
                    for itv in attempt_intervals:
                        try:
                            if df.empty:
                                cand = ticker.history(period='max', interval=itv)
                            else:
                                last_dt = df['datetime'].max()
                                cand = ticker.history(start=last_dt, interval=itv)
                        except Exception as e:
                            print(f"⚠️ yfinance.history error for {yf_symbol} interval {itv}: {e}")
                            cand = None

                        if cand is None or getattr(cand, 'empty', True):
                            # try next interval
                            continue

                        # Normalize to DataFrame with datetime column
                        try:
                            cand = cand.reset_index()
                        except Exception:
                            pass

                        time_col = 'Date' if 'Date' in getattr(cand, 'columns', []) else (
                            'Datetime' if 'Datetime' in getattr(cand, 'columns', []) else 'datetime')

                        # rename columns to expected names if present
                        rename_map = {}
                        if time_col in getattr(cand, 'columns', []):
                            rename_map[time_col] = 'datetime'
                        for k in ['Open', 'High', 'Low', 'Close', 'Volume']:
                            if k in getattr(cand, 'columns', []):
                                rename_map[k] = k.lower()

                        if rename_map:
                            cand = cand.rename(columns=rename_map)

                        # Ensure required columns exist
                        for col in ['open', 'high', 'low', 'close', 'volume']:
                            if col not in cand.columns:
                                cand[col] = None

                        if 'datetime' in cand.columns:
                            cand['datetime'] = pd.to_datetime(cand['datetime'], utc=True).dt.tz_localize(None)

                        # If fetched interval is finer than desired (e.g., we fetched 1h for H4), resample
                        if itv != interval and timeframe.upper() in ['H4', 'D1']:
                            try:
                                cand = resample_timeframe(cand[['datetime','open','high','low','close','volume']], timeframe.upper())
                            except Exception as e:
                                print(f"⚠️ Resample failed for {symbol} from {itv} to {timeframe}: {e}")

                        new_df = cand
                        # stop at first successful fetch
                        break

                    if new_df is not None and not getattr(new_df, 'empty', True):
                        # align columns
                        new_df = new_df[['datetime','open','high','low','close','volume']]

                        if df.empty:
                            df = new_df
                        else:
                            df = pd.concat([df, new_df])

                        df = df.drop_duplicates(subset=['datetime']).sort_values('datetime').reset_index(drop=True)

                        try:
                            df.to_parquet(cache_path)
                            print(f"✅ Cache updated for {symbol} ({len(df)} total bars)")
                        except Exception as e:
                            print(f"⚠️ Failed to write cache for {symbol}: {e}")
                except Exception as e:
                    print(f"❌ YFinance download failed for {symbol}: {e}")

            # final fallback: if df is still empty, try reading existing cache again or CSV fallback
            if df.empty:
                # try reading cache without lock
                if os.path.exists(cache_path):
                    try:
                        df = pd.read_parquet(cache_path)
                    except Exception:
                        df = pd.DataFrame()

            # If still empty, try resampling from a finer cached timeframe (e.g. H1 -> H4)
            if df.empty:
                try:
                    tf_minutes = { 'M1':1, 'M5':5, 'M15':15, 'H1':60, 'H4':240, 'D1':1440 }
                    target_min = tf_minutes.get(timeframe.upper())
                    if target_min is not None:
                        candidates = []
                        for f in os.listdir(config.CACHE_DIR):
                            if not f.startswith(f"{symbol}_") or not f.endswith('.parquet'):
                                continue
                            parts = f.replace('.parquet','').split('_')
                            if len(parts) != 2:
                                continue
                            cand_tf = parts[1]
                            cand_min = tf_minutes.get(cand_tf.upper())
                            if cand_min and cand_min < target_min:
                                candidates.append((cand_min, cand_tf, os.path.join(config.CACHE_DIR, f)))

                        # pick the finest available (smallest minutes)
                        if candidates:
                            candidates.sort()
                            cand_min, cand_tf, cand_path = candidates[0]
                            try:
                                base_df = pd.read_parquet(cand_path)
                                # ensure datetime index column
                                if 'datetime' not in base_df.columns and base_df.index.name in (None, 'datetime'):
                                    base_df = base_df.reset_index()
                                resampled = resample_timeframe(base_df[['datetime','open','high','low','close','volume']], timeframe.upper())
                                df = resampled
                                try:
                                    df.to_parquet(cache_path)
                                    print(f"ℹ️ Resampled {symbol} {cand_tf} -> {timeframe} and cached to {cache_path}")
                                except Exception as e:
                                    print(f"⚠️ Failed to write resampled cache for {symbol}: {e}")
                            except Exception as e:
                                print(f"⚠️ Resample fallback failed for {symbol} from {cand_tf}: {e}")
                except Exception:
                    pass

            if df.empty:
                # last resort: try CSV provider fallback
                try:
                    from .data_module import CSVDataProvider
                except Exception:
                    try:
                        # relative import fallback
                        from backend.data_module import CSVDataProvider
                    except Exception:
                        CSVDataProvider = None

                if CSVDataProvider:
                    try:
                        csvprov = CSVDataProvider()
                        df = csvprov.fetch_bars(symbol, timeframe, count, before=before)
                    except Exception:
                        pass

        if df.empty:
            return pd.DataFrame()

        # Handle slicing for the engine ( rispettando MAX_BARS_LIMIT )
        df = df.sort_values("datetime")
        if before:
            before_dt = pd.to_datetime(before)
            df = df[df["datetime"] < before_dt]

        return df.tail(count).reset_index(drop=True)


In [5]:
fetch_bars(symbol="EURUSD=X", timeframe= "1D", count= "100", before=None)

NameError: name 'config' is not defined

In [6]:
import os
import pandas as pd
import datetime
import yfinance as yf

# Local cache folder
CACHE_DIR = "market_cache"

# Symbols (Yahoo Finance format)
SYMBOLS = {
    "EURUSD": "EURUSD=X",
    "GBPUSD": "GBPUSD=X",
    "USDJPY": "JPY=X",
    "AUDUSD": "AUDUSD=X",
    "USDCAD": "CAD=X",
    "USDCHF": "CHF=X",
    "NZDUSD": "NZDUSD=X",
    "XAUUSD": "GC=F",      # Gold Futures
    "BTCUSD": "BTC-USD"
}

# Timeframes mapping
TIMEFRAMES = {
    "M5": "5m",
    "M15": "15m",
    "H1": "1h",
    "H4": "1h"   # we will resample to 4H
}


def resample_to_4h(df):
    df = df.set_index("datetime")

    resampled = df.resample("4H").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    })

    resampled = resampled.dropna().reset_index()
    return resampled


def download_market_data():

    os.makedirs(CACHE_DIR, exist_ok=True)

    for symbol, yf_symbol in SYMBOLS.items():

        print(f"\nDownloading {symbol}")

        ticker = yf.Ticker(yf_symbol)

        for tf, interval in TIMEFRAMES.items():

            cache_file = os.path.join(CACHE_DIR, f"{symbol}_{tf}.parquet")

            try:

                print(f"  → Fetching {tf}")

                # Get maximum available data
                df = ticker.history(period="max", interval=interval)

                if df.empty:
                    print("  ⚠ No data")
                    continue

                df = df.reset_index()

                time_col = "Datetime" if "Datetime" in df.columns else "Date"

                df = df.rename(columns={
                    time_col: "datetime",
                    "Open": "open",
                    "High": "high",
                    "Low": "low",
                    "Close": "close",
                    "Volume": "volume"
                })

                df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

                df = df[["datetime", "open", "high", "low", "close", "volume"]]

                # Resample H4
                if tf == "H4":
                    df = resample_to_4h(df)

                df = df.drop_duplicates("datetime").sort_values("datetime")

                df.to_parquet(cache_file)

                print(f"  ✅ Saved {len(df)} bars")

            except Exception as e:
                print(f"  ❌ Error: {e}")



In [10]:
import os
import pandas as pd
import yfinance as yf

CACHE_DIR = "market_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SYMBOLS = {
    "EURUSD": "EURUSD=X",
    "GBPUSD": "GBPUSD=X",
    "USDJPY": "JPY=X",
    "AUDUSD": "AUDUSD=X",
    "USDCAD": "CAD=X",
    "USDCHF": "CHF=X",
    "NZDUSD": "NZDUSD=X",
    "XAUUSD": "GC=F",
    "BTCUSD": "BTC-USD"
}

TIMEFRAMES = {
    "M5":  ("5m",  "60d"),
    "M15": ("15m", "60d"),
    "H1":  ("1h",  "730d")
}


def resample_4h(df):
    df = df.set_index("datetime")

    df = df.resample("4H").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    })

    return df.dropna().reset_index()


def download_market_data():

    for symbol, yf_symbol in SYMBOLS.items():

        print(f"\nDownloading {symbol}")

        ticker = yf.Ticker(yf_symbol)

        for tf, (interval, period) in TIMEFRAMES.items():

            print(f"  → Fetching {tf}")

            try:

                df = ticker.history(period=period, interval=interval)

                if df.empty:
                    print("  ⚠ No data")
                    continue

                df = df.reset_index()

                time_col = "Datetime" if "Datetime" in df.columns else "Date"

                df = df.rename(columns={
                    time_col: "datetime",
                    "Open": "open",
                    "High": "high",
                    "Low": "low",
                    "Close": "close",
                    "Volume": "volume"
                })

                df = df[["datetime","open","high","low","close","volume"]]

                df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

                df = df.drop_duplicates("datetime").sort_values("datetime")

                path = f"{CACHE_DIR}/{symbol}_{tf}.parquet"
                df.to_parquet(path)

                print(f"  ✅ Saved {len(df)} bars")

                # create H4 from H1
                if tf == "H1":

                    df4 = resample_4h(df)

                    path = f"{CACHE_DIR}/{symbol}_H4.parquet"
                    df4.to_parquet(path)

                    print(f"  ✅ Saved {len(df4)} H4 bars")

            except Exception as e:

                print(f"  ❌ Error: {e}")

download_market_data()

$EURUSD=X: possibly delisted; no price data found  (period=60d)
$EURUSD=X: possibly delisted; no price data found  (period=60d)
$EURUSD=X: possibly delisted; no price data found  (period=730d)
$GBPUSD=X: possibly delisted; no price data found  (period=60d)
$GBPUSD=X: possibly delisted; no price data found  (period=60d)
$GBPUSD=X: possibly delisted; no price data found  (period=730d)
$JPY=X: possibly delisted; no price data found  (period=60d)
$JPY=X: possibly delisted; no price data found  (period=60d)
$JPY=X: possibly delisted; no price data found  (period=730d)
$AUDUSD=X: possibly delisted; no price data found  (period=60d)
$AUDUSD=X: possibly delisted; no price data found  (period=60d)
$AUDUSD=X: possibly delisted; no price data found  (period=730d)
Failed to get ticker 'CAD=X' reason: 'RequestsCookieJar' object has no attribute 'jar'
$CAD=X: possibly delisted; no price data found  (period=60d)
$CAD=X: possibly delisted; no price data found  (period=60d)
$CAD=X: possibly delisted; n


  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data


In [9]:
download_market_data()

$EURUSD=X: possibly delisted; no price data found  (period=60d)
$EURUSD=X: possibly delisted; no price data found  (period=60d)
$EURUSD=X: possibly delisted; no price data found  (period=730d)
$GBPUSD=X: possibly delisted; no price data found  (period=60d)
$GBPUSD=X: possibly delisted; no price data found  (period=60d)
$GBPUSD=X: possibly delisted; no price data found  (period=730d)
$JPY=X: possibly delisted; no price data found  (period=60d)
$JPY=X: possibly delisted; no price data found  (period=60d)
$JPY=X: possibly delisted; no price data found  (period=730d)
$AUDUSD=X: possibly delisted; no price data found  (period=60d)
$AUDUSD=X: possibly delisted; no price data found  (period=60d)
$AUDUSD=X: possibly delisted; no price data found  (period=730d)
Failed to get ticker 'CAD=X' reason: 'RequestsCookieJar' object has no attribute 'jar'
$CAD=X: possibly delisted; no price data found  (period=60d)
$CAD=X: possibly delisted; no price data found  (period=60d)
$CAD=X: possibly delisted; n


  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data

  → Fetching M5
  ⚠ No data
  → Fetching M15
  ⚠ No data
  → Fetching H1
  ⚠ No data


In [3]:
import yfinance as yf

df = yf.download(
    "BTC-USD",
    period="60d",
    interval="1h",
    progress=False
)

# Clean dataframe
df.columns = df.columns.get_level_values(0)
df.columns = df.columns.str.lower()
df = df.reset_index()

print(df.tail())

/tmp/ipykernel_300500/1763280227.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(


Price                  Datetime         close          high           low  \
1423  2026-03-06 07:00:00+00:00  71077.156250  71162.750000  70495.625000   
1424  2026-03-06 08:00:00+00:00  70762.578125  71060.335938  70762.578125   
1425  2026-03-06 09:00:00+00:00  70565.007812  70785.617188  70353.242188   
1426  2026-03-06 10:00:00+00:00  70551.640625  70765.609375  70503.109375   
1427  2026-03-06 11:00:00+00:00  70540.937500  70567.984375  70430.210938   

Price          open      volume  
1423   70616.273438   413925376  
1424   71060.335938  1133920256  
1425   70762.046875   285122560  
1426   70566.250000  1198419968  
1427   70540.765625   151470080  


In [2]:
pip uninstall eventlet -y


Found existing installation: eventlet 0.40.4
Uninstalling eventlet-0.40.4:
  Successfully uninstalled eventlet-0.40.4
Note: you may need to restart the kernel to use updated packages.
